# 04 - Diagnóstico de los datos del SAIH

Verificación de la estructura y consistencia de los ficheros recibidos del SAIH del
Miño-Sil, previa a su procesamiento. El notebook documenta cuatro cuestiones detectadas
durante la inspección de los datos:

1. **Diccionario de variables.** Los ficheros contienen más códigos de variable de los
   inicialmente identificados, incluyendo dos parámetros de calidad del agua adicionales.
2. **Inventario de estaciones.** Estaciones presentes en los datos y disponibilidad de
   sus coordenadas.
3. **Identificador E03S.** Serie que no corresponde a ningún embalse del maestro.
4. **Porcentajes de llenado superiores al 100%.** Origen del valor máximo anómalo
   detectado en el análisis exploratorio para el embalse E003.

## Salidas

- `data/processed/diccionario_variables_saih.parquet`: descripción y unidades oficiales
  de cada código de variable, extraídas de las cabeceras de los propios ficheros.

## 1. Configuración

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("..")
from src.saih import PATRON_COL, id_estacion, num_saih, normalizar_id_embalse

DIR_RAW_ANUARIO = Path("../data/raw/anuario")
CSV_KWARGS = {"sep": ";", "encoding": "latin-1"}
DIR_RAW_SAIH = Path("../data/raw/saih")
DIR_PROCESSED = Path("../data/processed")

# Hojas de datos de los tres ficheros recibidos
HOJAS = [
    ("Datos_Embalses.xlsx", "Datos_Embalses"),
    ("Datos_Estaciones.xlsx", "Datos_Nivel"),
    ("Datos_Estaciones.xlsx", "Datos_Caudal"),
    ("Datos_Estaciones.xlsx", "Datos_Precipitación"),
    ("Datos_Estaciones.xlsx", "Datos_TempAmb"),
    ("Datos_Estaciones_Calidad.xlsx", "Datos_Calidad"),
]

maestro = pd.read_parquet(DIR_PROCESSED / "maestro_embalses.parquet")
print(f"Embalses en el maestro: {len(maestro)}")

Embalses en el maestro: 35


## 2. Diccionario de variables

Las hojas de datos del SAIH emplean una cabecera de tres filas: código de columna,
descripción de la variable y unidad de medida. El código de columna combina el
identificador de estación con el código de variable, en dos formatos distintos según el
tipo de estación (`A002_AINRIO1` para estaciones en río, `E001MACVEMBA` para estaciones
en embalse).

Se extrae de estas cabeceras el diccionario de variables, que constituye la
documentación oficial de los datos recibidos.

In [2]:
filas = []
for fichero, hoja in HOJAS:
    cab = pd.read_excel(DIR_RAW_SAIH / fichero, sheet_name=hoja, header=None, nrows=3)
    for i in range(1, cab.shape[1]):
        m = PATRON_COL.match(str(cab.iloc[0, i]).strip().split(" ")[0])
        if m:
            filas.append({
                "hoja": hoja,
                "codigo_variable": m.group(2).upper(),
                "descripcion": str(cab.iloc[1, i]).strip(),
                "unidad": str(cab.iloc[2, i]).strip(),
            })

diccionario = (
    pd.DataFrame(filas)
    .drop_duplicates(subset=["hoja", "codigo_variable"])
    .sort_values(["hoja", "codigo_variable"])
    .reset_index(drop=True)
)

diccionario.to_parquet(DIR_PROCESSED / "diccionario_variables_saih.parquet", index=False)
print(diccionario.to_string(index=False))

               hoja codigo_variable                       descripcion unidad
      Datos_Calidad         AIA3ATS               Amonio medio diario   mg/L
      Datos_Calidad         AIA6AFS             Fosfatos medio diario   mg/L
      Datos_Calidad         AIMOMOS     Materia orgánica media diaria    m-1
      Datos_Calidad         AIMPCTS        Conductividad media diaria  µS/cm
      Datos_Calidad         AIMPO2S     Oxígeno disuelto medio diario   mg/L
      Datos_Calidad         AIMPPHS                   pH medio diario   u.pH
      Datos_Calidad         AIMPTTS Temperatura del agua media diaria     °C
      Datos_Calidad         AITUTUS             Turbidez media diaria    NTU
       Datos_Caudal          ACQRIO                        Caudal río (m3/s)
       Datos_Caudal          AIQRIO                        Caudal río (m3/s)
     Datos_Embalses         ACAPORT                 Caudal aportación (m3/s)
     Datos_Embalses         ACQTSAL                     Caudal salida (m3/s)

## 3. Inventario de estaciones

Se identifican las estaciones presentes en cada hoja de datos y se comprueba que todas
las columnas resultan parseables, de modo que ningún código de variable no contemplado
provoque la pérdida silenciosa de estaciones.

In [3]:
def ids_de_hoja(fichero, hoja):
    """Identificadores de estación presentes en las cabeceras de una hoja."""
    cols = pd.read_excel(DIR_RAW_SAIH / fichero, sheet_name=hoja, nrows=0).columns
    return {i for i in (id_estacion(c) for c in cols[1:]) if i}

inventario = {hoja: ids_de_hoja(fichero, hoja) for fichero, hoja in HOJAS}

for hoja, ids in inventario.items():
    print(f"{hoja:<22} {len(ids):>4} estaciones")

# Control: ninguna columna puede quedar sin parsear
sin_parsear = []
for fichero, hoja in HOJAS:
    cols = pd.read_excel(DIR_RAW_SAIH / fichero, sheet_name=hoja, nrows=0).columns
    sin_parsear += [c for c in cols[1:] if id_estacion(c) is None]

assert not sin_parsear, f"Códigos de variable no reconocidos: {sin_parsear}"
print("\nTodas las columnas parseadas correctamente.")

Datos_Embalses           36 estaciones
Datos_Nivel              61 estaciones
Datos_Caudal             55 estaciones
Datos_Precipitación      93 estaciones
Datos_TempAmb            96 estaciones
Datos_Calidad            18 estaciones

Todas las columnas parseadas correctamente.


### Cobertura de coordenadas

Las coordenadas de las estaciones proceden de las hojas de identificación de
`Datos_Estaciones.xlsx` (estaciones en río y meteorológicas) y `Datos_Embalses.xlsx`
(estaciones en embalse). Se comprueba qué estaciones presentes en los datos carecen de
coordenadas conocidas.

Las estaciones ubicadas en embalses emplean en las hojas de datos una codificación de
tres dígitos (`E005A`, `E016A`) frente a la de dos dígitos más letra de las hojas de
identificación (`E05A`, `E16A`), por lo que la correspondencia se establece sobre la
parte numérica del identificador.

In [4]:
def cargar_identificacion(fichero):
    """Hoja de identificación de estaciones, con cabecera de dos niveles."""
    df = pd.read_excel(DIR_RAW_SAIH / fichero,
                       sheet_name="Identificación Estaciones", skiprows=[1])
    df.columns = ["ID", "Nombre", "Municipio", "Provincia", "Sistema", "X", "Y"]
    df["ID"] = df["ID"].astype(str).str.strip()
    return df[df["ID"].ne("nan")]

ident = pd.concat([
    cargar_identificacion("Datos_Estaciones.xlsx"),
    cargar_identificacion("Datos_Embalses.xlsx"),
], ignore_index=True).drop_duplicates(subset="ID")

# Correspondencia por parte numérica para las estaciones en embalse
ident["num"] = ident["ID"].apply(num_saih)
num_a_id = ident.dropna(subset=["num"]).set_index("num")["ID"].to_dict()

todas = set().union(*inventario.values())
sin_coords = {
    e for e in todas
    if e not in set(ident["ID"]) and num_a_id.get(num_saih(e)) is None
}

print(f"Estaciones con coordenadas: {len(ident)}")
print(f"Estaciones distintas en los datos: {len(todas)}")
print(f"Sin coordenadas asignables: {sorted(sin_coords)}")

Estaciones con coordenadas: 115
Estaciones distintas en los datos: 124
Sin coordenadas asignables: ['A004', 'A021']


## 4. Identificación de la serie E03S

El identificador `E03S` aparece en la hoja de datos de embalses pero no figura en la
hoja de identificación ni en el maestro de embalses, por lo que carece de coordenadas y
de capacidad asociada.

La hipótesis de partida es que se trata de una medida agregada del sistema formado por
los embalses de Las Rozas (E003) y Matalavilla (E05A), en coherencia con el registro
`E003-E005`, denominado *ROZAS, LAS - MATALAVILLA (SIST.)*, presente en el catálogo del
Anuario de Aforos. Se verifica comparando la serie agregada con la suma de las
individuales.

In [5]:
emb = pd.read_excel(DIR_RAW_SAIH / "Datos_Embalses.xlsx",
                    sheet_name="Datos_Embalses", skiprows=[1, 2])
emb = emb.rename(columns={emb.columns[0]: "fecha"})
emb["fecha"] = pd.to_datetime(emb["fecha"])

# Cobertura temporal de cada serie
for col in ["E003MACVEMBA", "E05AMACVEMBA", "E03SMAIVEMBA"]:
    s = emb.loc[emb[col].notna(), ["fecha", col]]
    print(f"{col}: {len(s):>5} días | {s['fecha'].min():%Y-%m} a {s['fecha'].max():%Y-%m}"
          f" | máx {s[col].max():.2f} hm3")

# Comparación sobre el periodo en que las tres series coexisten
tres = emb[["fecha", "E003MACVEMBA", "E05AMACVEMBA", "E03SMAIVEMBA"]].dropna()
dif = tres["E03SMAIVEMBA"] - (tres["E003MACVEMBA"] + tres["E05AMACVEMBA"])

print(f"\nDías comparables: {len(tres)}")
print(f"Diferencia mediana E03S - (E003 + E05A): {dif.median():.3f} hm3")
print(f"Diferencia máxima absoluta: {dif.abs().max():.3f} hm3")

E003MACVEMBA:  4677 días | 2009-01 a 2024-05 | máx 44.09 hm3
E05AMACVEMBA:  4616 días | 2011-10 a 2024-05 | máx 64.33 hm3
E03SMAIVEMBA:  8907 días | 2000-01 a 2024-05 | máx 91.47 hm3

Días comparables: 4616
Diferencia mediana E03S - (E003 + E05A): 0.000 hm3
Diferencia máxima absoluta: 35.790 hm3


### Conclusión sobre E03S

La diferencia entre la serie agregada y la suma de las individuales es nula, lo que
confirma que `E03S` corresponde al conjunto Rozas-Matalavilla.

La serie agregada cubre el periodo completo de estudio, mientras que las series
individuales comienzan en 2009 (E003) y 2011 (E05A).

## 5. Porcentajes de llenado superiores al 100%

El análisis exploratorio inicial detectó un porcentaje de llenado máximo del 157,46% en
el embalse E003, muy por encima del resto de anomalías observadas. Se investiga su
origen.

In [6]:
CAPACIDAD_E003 = maestro.loc[maestro["ID_SAIH"] == "E003", "Capacidad_hm3"].iloc[0]
pct = emb["E003MACVEMBA"] / CAPACIDAD_E003 * 100

print(f"Capacidad de referencia (CHMS): {CAPACIDAD_E003:.2f} hm3")
print(f"Días con dato: {pct.notna().sum()}")
print(f"Días por encima del 100%: {(pct > 100).sum()}")
print(f"Días por encima del 120%: {(pct > 120).sum()}")

print("\nDistribución mensual de los días por encima del 100%:")
print(emb.loc[pct > 100, "fecha"].dt.to_period("M").value_counts().sort_index().to_string())

Capacidad de referencia (CHMS): 28.00 hm3
Días con dato: 4677
Días por encima del 100%: 54
Días por encima del 120%: 31

Distribución mensual de los días por encima del 100%:
fecha
2009-01     6
2011-10    31
2012-05     3
2013-01     2
2013-02     1
2013-03     2
2013-04     2
2016-03     1
2018-06     1
2023-01     1
2024-03     3
2024-04     1
Freq: M


In [7]:
# El grueso de los excesos se concentra en octubre de 2011, mes en que comienza la
# medida individual de Matalavilla. Se comprueba si la serie de E003 reproduce en esas
# fechas el valor del sistema agregado.
oct11 = emb[emb["fecha"].dt.to_period("M") == "2011-10"]
coincidencias = (oct11["E003MACVEMBA"] == oct11["E03SMAIVEMBA"]).sum()

print(f"Días de octubre de 2011: {len(oct11)}")
print(f"Días en que E003 coincide exactamente con E03S: {coincidencias}")
print()
print(oct11[["fecha", "E003MACVEMBA", "E05AMACVEMBA", "E03SMAIVEMBA"]].head(10).to_string(index=False))

Días de octubre de 2011: 31
Días en que E003 coincide exactamente con E03S: 31

     fecha  E003MACVEMBA  E05AMACVEMBA  E03SMAIVEMBA
2011-10-01         44.09         35.79         44.09
2011-10-02         44.03         35.76         44.03
2011-10-03         44.00         35.73         44.00
2011-10-04         43.97         35.71         43.97
2011-10-05         43.93         35.68         43.93
2011-10-06         43.87         35.65         43.87
2011-10-07         43.87         35.62         43.87
2011-10-08         43.94         35.68         43.94
2011-10-09         43.91         35.68         43.91
2011-10-10         43.88         35.65         43.88


In [8]:
# Referencias de nivel del Anuario, enlazadas al identificador del SAIH mediante la
# tabla de correspondencia construida en el notebook 02.
crosswalk = pd.read_parquet(DIR_PROCESSED / "crosswalk_anuario_saih.parquet")
emb_anuario = pd.read_csv(DIR_RAW_ANUARIO / "embalse.csv", **CSV_KWARGS)

referencias = (crosswalk[["ref_ceh", "ID_SAIH"]]
               .merge(emb_anuario[["ref_ceh", "mnne", "mna"]], on="ref_ceh", how="left"))

# El valor 0 en las columnas de nivel indica ausencia de dato, no cota cero
referencias[["mnne", "mna"]] = referencias[["mnne", "mna"]].replace(0, pd.NA)

def columnas_por_variable(columnas, grupo):
    """Mapa identificador de embalse -> columna, para un conjunto de códigos de variable."""
    salida = {}
    for c in columnas:
        m = PATRON_COL.match(str(c).strip())
        if m and m.group(2).upper() in grupo:
            est = m.group(1).upper()
            est = est if est in set(maestro["ID_SAIH"]) else normalizar_id_embalse(est)
            if est:
                salida[est] = c
    return salida

col_nivel = columnas_por_variable(emb.columns[1:], {"MAINEMBA", "AINEMBA"})
col_volumen = columnas_por_variable(emb.columns[1:], {"MACVEMBA", "MAIVEMBA", "ACVEMBA"})

comp = pd.DataFrame({
    "ID_SAIH": list(col_nivel),
    "nivel_max_obs": [emb[c].max() for c in col_nivel.values()],
})
comp["volumen_max_obs"] = comp["ID_SAIH"].map({k: emb[v].max() for k, v in col_volumen.items()})
comp = (comp.merge(referencias[["ID_SAIH", "mnne", "mna"]], on="ID_SAIH", how="left")
            .merge(maestro[["ID_SAIH", "Nombre_SAIH", "Capacidad_hm3"]], on="ID_SAIH", how="left"))

comp["pct_max"] = comp["volumen_max_obs"] / comp["Capacidad_hm3"] * 100
comp["sobre_mnne_m"] = comp["nivel_max_obs"] - comp["mnne"]

resultado = comp[comp["mnne"].notna()].sort_values("pct_max", ascending=False)

print(f"Embalses con referencia de nivel disponible: {len(resultado)} de {len(comp)}\n")
print("=== Embalses que superan el 100% de llenado ===")
print(resultado[resultado["pct_max"] > 100]
      [["ID_SAIH", "Nombre_SAIH", "pct_max", "nivel_max_obs", "mnne", "mna", "sobre_mnne_m"]]
      .round(2).to_string(index=False))

print("\n=== Desviación respecto al nivel normal en el resto ===")
resto = resultado[resultado["pct_max"] <= 100]
print(f"  mediana {resto['sobre_mnne_m'].median():.2f} m | "
      f"mínimo {resto['sobre_mnne_m'].min():.2f} m | máximo {resto['sobre_mnne_m'].max():.2f} m")

anomalos = resto[resto["sobre_mnne_m"].abs() > 10]
if len(anomalos):
    print("\nDesviaciones superiores a 10 m (posible discrepancia entre fuentes):")
    print(anomalos[["ID_SAIH", "Nombre_SAIH", "nivel_max_obs", "mnne", "sobre_mnne_m"]]
          .round(2).to_string(index=False))

Embalses con referencia de nivel disponible: 33 de 35

=== Embalses que superan el 100% de llenado ===
ID_SAIH Nombre_SAIH  pct_max  nivel_max_obs  mnne    mna  sobre_mnne_m
   E003    As Rozas   157.46         959.46 959.5   <NA>         -0.04
   E022   Guistolas   105.50         699.85 700.0  700.5         -0.15
   E011  Peñarrubia   105.17         394.45 394.4  394.9          0.05
   E013  San Martin   105.10         290.26 290.0   <NA>          0.26
   E029   San Pedro   100.70         129.87 130.0  132.0         -0.13
   E028   Vilasouto   100.15         473.25 473.0  474.2          0.25
   E027 San Esteban   100.02         228.98 229.0  229.0         -0.02

=== Desviación respecto al nivel normal en el resto ===
  mediana -0.12 m | mínimo -3.78 m | máximo 77.39 m

Desviaciones superiores a 10 m (posible discrepancia entre fuentes):
ID_SAIH Nombre_SAIH  nivel_max_obs  mnne  sobre_mnne_m
   E021   Chandrexa         987.39 910.0         77.39
